# [센서 실습 29] 터치 센서 – TTP223 정전식 터치 모듈

정전식 터치 모듈 **TTP223**의 출력을 읽어 **짧게 누르기 · 두 번 누르기 · 길게 누르기**를 구분하고, 각각 다른 기능(조명 켜기/끄기, 밝기 변경, 전원 끄기)에 연결합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
스마트폰 화면처럼 **손가락이 닿으면 전기 용량(정전용량)이 늘어나는** 원리를 이용합니다.
- 사람 몸은 전기를 저장하는 큰 그릇 역할 → 손가락이 패드에 가까워지면 패드의 정전용량 증가
- TTP223 칩이 이 변화를 감지해 **디지털 출력**: 기본 설정에서 **터치하면 HIGH(1)**
- 아크릴·유리 몇 mm 너머에서도 감지 → 방수 버튼, 숨은 스위치로 사용
- 모듈의 A·B 점퍼로 동작 방식 선택 (출력 반전, 토글 모드)

입력 장치에서 중요한 것은 **시간 정보**입니다.

| 동작 | 판단 기준 |
|---|---|
| 짧게 누르기 (탭) | 누른 시간 < 0.5초 |
| 두 번 누르기 | 탭 두 번 사이 간격 < 0.4초 |
| 길게 누르기 | 누른 시간 ≥ 1초 |

### 부품 사양
| 항목 | 내용 |
|---|---|
| 감지 | 정전식 (패드 위 최대 수 mm) |
| 응답 시간 | 약 60ms (저속 모드 220ms) |
| 출력 | 디지털 (터치 시 HIGH, 점퍼로 변경) |
| 전원 | 2.0 ~ 5.5V |

## 2. 라즈베리파이 연결 방법
**준비물**: TTP223 모듈, 점퍼선 3개, (조명 역할) LED + 330Ω

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| TTP223 VCC / GND | 3.3V / GND |  |
| TTP223 I/O | GPIO22 (15번 핀) | 터치 시 1 |
| LED(+) – 330Ω | GPIO18 (12번 핀) | PWM으로 밝기 조절 |



**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
0/1 신호 목록을 **누른 구간**으로 바꾸고, 길이와 간격으로 동작을 구분하는 함수입니다.

In [ ]:
TAP_MAX, LONG_MIN, DOUBLE_GAP = 0.5, 1.0, 0.4      # 초

def presses_from_signal(times, signal):
    """0/1 신호 → [(누른 시각, 뗀 시각), ...]"""
    out, start = [], None
    for t, s in zip(times, signal):
        if s and start is None: start = t
        if not s and start is not None:
            out.append((start, t)); start = None
    return out

def classify(presses):
    """누른 구간 목록 → 동작 목록 [(시각, '탭'/'더블탭'/'길게'), ...]"""
    events, i = [], 0
    while i < len(presses):
        s, e = presses[i]
        if e - s >= LONG_MIN:
            events.append((s, "길게")); i += 1; continue
        if i + 1 < len(presses) and presses[i + 1][0] - e < DOUBLE_GAP and presses[i + 1][1] - presses[i + 1][0] < TAP_MAX:
            events.append((s, "더블탭")); i += 2; continue
        if e - s < TAP_MAX:
            events.append((s, "탭"))
        i += 1
    return events

### 4-2. 실제 센서 클래스
`DigitalInputDevice`로 터치 상태(0/1)를 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealTTP223:
    def __init__(self, pin=22):
        from gpiozero import DigitalInputDevice
        self.dev = DigitalInputDevice(pin, pull_up=None, active_state=True)

    def read(self):
        return {"touch": int(self.dev.value)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 시각(초) | 가정한 손동작 | 누른 시간 |
|---|---|---|
| 1.0 | 짧게 누르기 | 0.20초 |
| 3.0 / 3.35 | 두 번 누르기 | 0.15초씩, 간격 0.2초 |
| 6.0 | 길게 누르기 | 1.5초 |
| 9.0 | 짧게 누르기 | 0.25초 |
| 11.0 / 11.30 | 두 번 누르기 | 0.15초씩 |
| 13.0 | 애매하게 누름 | 0.7초 (탭도 길게도 아님 → 무시) |

TTP223의 응답 지연(약 60ms)만큼 늦게 바뀌도록 만들었습니다.

In [ ]:
class VirtualTTP223:
    PRESSES = [(1.0, 1.2), (3.0, 3.15), (3.35, 3.5), (6.0, 7.5), (9.0, 9.25), (11.0, 11.15), (11.3, 11.45), (13.0, 13.7)]
    DELAY = 0.06

    def __init__(self, dt=0.05, seed=29):
        self.dt, self.t = dt, 0.0

    def read(self):
        t = self.t - self.DELAY
        touch = int(any(a <= t < b for a, b in self.PRESSES))
        self.t += self.dt
        return {"touch": touch}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualTTP223(dt=0.05)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
20Hz(0.05초 간격)로 15초 동안 읽습니다. 짧은 탭도 놓치지 않으려면 측정 간격이 누르는 시간보다 훨씬 짧아야 합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 300               # 측정 횟수
INTERVAL = 0.05               # 측정 간격(초)

sensor = RealTTP223() if USE_REAL_SENSOR else VirtualTTP223(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
동작을 구분한 뒤 기능에 연결합니다.

| 동작 | 기능 |
|---|---|
| 탭 | 조명 켜기/끄기 |
| 더블탭 | 밝기 단계 변경 (30% → 60% → 100%) |
| 길게 | 모든 기기 끄기 |

In [ ]:
presses = presses_from_signal(df.time_s, df.touch)
events = classify(presses)
light, levels, lv = False, [30, 60, 100], 2
for t, kind in events:
    if kind == "탭":
        light = not light
    elif kind == "더블탭":
        lv = (lv + 1) % 3; light = True
    elif kind == "길게":
        light = False
    print(f"{t:5.2f}초  {kind:4s} → 조명 {'켜짐 ' + str(levels[lv]) + '%' if light else '꺼짐'}")
print(f"\n누른 횟수 {len(presses)}번 → 인식한 동작 {len(events)}개 (애매한 0.7초 누름은 무시)")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.step(df.time_s, df.touch, "k-", where="post"); ax.fill_between(df.time_s, df.touch, step="post", color="lightgray")
for t, kind in events:
    ax.annotate(kind, (t, 1.05), ha="left", fontsize=9)
ax.set_ylim(-0.1, 1.4); ax.set_yticks([0, 1]); ax.set_yticklabels(["뗌", "터치"])
ax.set_title("TTP223 터치 입력과 동작 인식"); ax.set_xlabel("시간 (초)"); ax.grid(alpha=0.3)
plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "터치센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "TTP223 정전식 터치 모듈",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/touch", payload)
# client.disconnect()

## 직접 해 보기
- 측정 간격을 0.2초로 늘리면 짧은 탭을 놓치는지 확인해 보세요.
- `DOUBLE_GAP`을 0.1초로 줄이면 더블탭이 어떻게 인식되나요?
- 세 번 누르기(트리플 탭)를 인식하도록 `classify`를 고쳐 보세요.